# Mini-Evaluation (S1 vs S2 vs S3 vs S4)
This notebook runs a small ablation evaluation with 2 queries per FA-Type.

In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path

# Resolve project root regardless of Jupyter's starting CWD
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("Could not locate project root (pyproject.toml)")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

from src.common.utils import setup_logging
from src.common.ingestion import download_all_filings
from src.systems.rag_monolith.pipeline import MonolithRAGPipeline
from src.systems.rag_agent.pipeline import AgentRAGPipeline
from src.systems.long_context.pipeline import LongContextPipeline
from src.systems.multi_agent.pipeline import MultiAgentPipeline

setup_logging()
GOLD_CSV = PROJECT_ROOT / "data" / "gold_standard" / "gold_standard_v3.csv"

In [ ]:
# Load 2 queries of each type
df_gs = pd.read_csv(GOLD_CSV, sep=";")
df_sample = (
    df_gs.groupby("fa_type", group_keys=False)
    .sample(n=2, random_state=42)
    .reset_index(drop=True)
)
queries = df_sample["query"].tolist()
display(df_sample[["id", "fa_type", "query"]])
print(f"Loaded {len(df_sample)} queries from {GOLD_CSV.relative_to(PROJECT_ROOT)}")


In [ ]:
# Load SEC filings (downloads/parses if not cached)
filings = download_all_filings()
print(f"Loaded {len(filings)} filings.")

In [ ]:
# Initialize and build pipelines
print("Building S1 (RAG Monolith)...")
s1 = MonolithRAGPipeline()
s1.build(filings)

print("Building S2 (Agent RAG)...")
s2 = AgentRAGPipeline()
s2.build(filings)

print("Building S3 (Long Context)...")
s3 = LongContextPipeline()
s3.build(filings)

print("Building S4 (Multi-Agent)...")
s4 = MultiAgentPipeline()
s4.build(filings)


In [ ]:
# Run evaluation
results = []

for i, row in df_sample.iterrows():
    q = row['query']
    fa_type = row['fa_type']
    print(f"\n{'='*50}\nQuery [{fa_type}]: {q}")
    
    r1 = s1.query(q)
    print(f"[S1] {r1.metrics.token_usage.total_tokens} tokens | Ans: {r1.answer[:100]}...")
    
    r2 = s2.query(q)
    print(f"[S2] {r2.metrics.token_usage.total_tokens} tokens | Ans: {r2.answer[:100]}...")
    
    r3 = s3.query(q)
    print(f"[S3] {r3.metrics.token_usage.total_tokens} tokens | Ans: {r3.answer[:100]}...")
    
    r4 = s4.query(q)
    print(f"[S4] {r4.metrics.token_usage.total_tokens} tokens | Ans: {r4.answer[:100]}...")
    
    results.append({
        'id': row['id'],
        'fa_type': fa_type,
        'query': q,
        'S1_tokens': r1.metrics.token_usage.total_tokens,
        'S1_ans': r1.answer,
        'S2_tokens': r2.metrics.token_usage.total_tokens,
        'S2_ans': r2.answer,
        'S3_tokens': r3.metrics.token_usage.total_tokens,
        'S3_ans': r3.answer,
        'S4_tokens': r4.metrics.token_usage.total_tokens,
        'S4_ans': r4.answer
    })


In [ ]:
# Save and display results
RESULTS_CSV = PROJECT_ROOT / "data" / "results" / "mini_eval_s1_s4_results.csv"
RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
df_results = pd.DataFrame(results)
df_results.to_csv(RESULTS_CSV, index=False, sep=";")
display(df_results[["fa_type", "S1_tokens", "S2_tokens", "S3_tokens", "S4_tokens"]])
print(f"Saved: {RESULTS_CSV.relative_to(PROJECT_ROOT)}")
